# MLB Data Starter Notebook

This notebook is a shared starting template for querying datasets produced by `mlb-data-pipeline`.

It automatically locates the latest local snapshot under `mlb-data-pipeline/output/baseball` and loads aggregate tables for analysis.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.dataset as ds

In [ ]:
def find_baseball_output_root() -> Path:
    marker = Path('mlb-data-pipeline/output/baseball/latest.json')

    search_roots = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/home/jovyan/work'),
        Path('/home/jovyan/assignments'),
    ]

    for root in search_roots:
        candidate = root / marker
        if candidate.exists():
            return candidate.parent

    raise FileNotFoundError(
        'Could not find mlb-data-pipeline/output/baseball/latest.json. '
        'Run the pipeline locally or mount data into this environment first.'
    )


output_root = find_baseball_output_root()
latest_path = output_root / 'latest.json'

with latest_path.open('r', encoding='utf-8') as f:
    latest = json.load(f)

snapshot_id = latest['snapshot_id']
snapshot_dir_candidates = [
    output_root / snapshot_id,
    output_root / 'snapshots' / snapshot_id,
]
snapshot_dir = next((p for p in snapshot_dir_candidates if p.exists()), None)
if snapshot_dir is None:
    raise FileNotFoundError(f'Could not locate local snapshot directory for {snapshot_id}.')

manifest_path = snapshot_dir / 'manifest.json'
with manifest_path.open('r', encoding='utf-8') as f:
    manifest = json.load(f)

print(f'Loaded snapshot: {snapshot_id}')
print(f'Snapshot directory: {snapshot_dir}')
print(f'Window: {latest.get("window_start_date")} to {latest.get("window_end_date")}')

In [ ]:
aggregates_dir = snapshot_dir / 'aggregates'

player_season = pd.read_parquet(aggregates_dir / 'player_season_metrics.parquet')
player_career = pd.read_parquet(aggregates_dir / 'player_career_metrics.parquet')
team_season = pd.read_parquet(aggregates_dir / 'team_season_metrics.parquet')
team_career = pd.read_parquet(aggregates_dir / 'team_career_metrics.parquet')

print('player_season rows:', len(player_season))
print('player_career rows:', len(player_career))
print('team_season rows:', len(team_season))
print('team_career rows:', len(team_career))

In [ ]:
season_col = 'season' if 'season' in player_season.columns else 'game_year'
target_season = int(player_season[season_col].max())

leaders = (
    player_season.loc[player_season[season_col] == target_season]
    .sort_values('ops', ascending=False)
    .head(20)
    [[c for c in ['player_name', season_col, 'pa', 'avg', 'obp', 'slg', 'ops'] if c in player_season.columns]]
)

leaders

## Sample Queries: Team Stats

The next cells show a few common team-level queries on the latest season in the snapshot.

In [ ]:
team_season_col = 'season' if 'season' in team_season.columns else 'game_year'
latest_team_season = int(team_season[team_season_col].max())

candidate_team_cols = ['team_name', 'team', 'team_abbrev', 'home_team']
team_col = next((c for c in candidate_team_cols if c in team_season.columns), None)

display_cols = [c for c in [team_col, team_season_col, 'g', 'pa', 'avg', 'obp', 'slg', 'ops'] if c and c in team_season.columns]

latest_team_stats = (
    team_season.loc[team_season[team_season_col] == latest_team_season, display_cols]
    .sort_values('ops', ascending=False)
    .head(15)
)

latest_team_stats

In [ ]:
team_to_inspect = 'Los Angeles Dodgers'

if team_col is None:
    raise ValueError('Could not find a team identifier column in team_season_metrics.')

team_history = (
    team_season.loc[team_season[team_col] == team_to_inspect]
    .sort_values(team_season_col)
    [[c for c in [team_col, team_season_col, 'g', 'pa', 'avg', 'obp', 'slg', 'ops'] if c in team_season.columns]]
)

team_history.tail(10)

In [ ]:
recent_seasons = sorted(team_season[team_season_col].dropna().unique())[-3:]

team_ops_trend = (
    team_season.loc[team_season[team_season_col].isin(recent_seasons)]
    .groupby(team_season_col, as_index=False)['ops']
    .mean()
    .rename(columns={'ops': 'league_avg_team_ops'})
)

team_ops_trend

In [ ]:
plot_df = team_ops_trend.sort_values(team_season_col)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(plot_df[team_season_col], plot_df['league_avg_team_ops'], marker='o')
ax.set_title('League Average Team OPS (Recent Seasons)')
ax.set_xlabel('Season')
ax.set_ylabel('OPS')
ax.grid(alpha=0.25)
plt.tight_layout()
plot_df

In [ ]:
detail_dir = snapshot_dir / 'detail'
if detail_dir.exists():
    detail_ds = ds.dataset(detail_dir, format='parquet', partitioning='hive')
    detail_preview = detail_ds.to_table(columns=['game_date', 'player_name', 'events', 'launch_speed']).to_pandas().head(10)
    detail_preview
else:
    print('No detail directory found in this snapshot.')

## Next Steps

- Add your own filters (season, player, team).
- Save charts in new cells below this section.
- Keep this file as your baseline template and duplicate it for analyses.